# EKF-Adaptive Pruning: Stage 1 Baseline Training

**目标**: ResNet18-CIFAR baseline -> 93%+ test accuracy

**预计时间**: T4 约 2-3 小时 (100 epochs)

---

## 运行顺序
1. 挂载Drive (如果要保存checkpoint到Drive)
2. clone/上传项目代码
3. 运行 `train_base.py`
4. 跑完去看 `best_acc`

## Step 1: 检查GPU

In [ ]:
!nvidia-smi

## Step 2: 挂载Drive (可选, 但强烈建议)

这样 checkpoint 和数据不会在 runtime 重启时丢失.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 项目根目录 (在Drive里建一个文件夹)
import os
PROJECT_ROOT = '/content/drive/MyDrive/ekf_adaptive_pruning'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
print(f'[cwd] {os.getcwd()}')

## Step 3: 上传项目代码

两种方式:
- **A. 手动上传**: 把 `models/resnet18_cifar.py` 和 `src/train_base.py` 直接拖到Drive对应位置
- **B. git clone**: 如果已经推到GitHub, `!git clone <repo_url>`

目录结构应该是:
```
ekf_adaptive_pruning/
├── models/
│   ├── __init__.py          # 空文件
│   └── resnet18_cifar.py
├── src/
│   ├── __init__.py          # 空文件
│   └── train_base.py
├── data/                    # CIFAR-10自动下载到这里
└── checkpoints/             # 自动创建
```

In [ ]:
# 创建空的 __init__.py (让 Python 能 import models/src 作为模块)
!touch models/__init__.py src/__init__.py
!ls -la

## Step 4: Smoke test - 先跑一下模型能不能正向传播

In [ ]:
import sys
sys.path.insert(0, '.')  # 让 Python 能找到 models/ 和 src/

import torch
from models.resnet18_cifar import resnet18_cifar

model = resnet18_cifar(num_classes=10).cuda()
x = torch.randn(4, 3, 32, 32).cuda()
y = model(x)
print(f'Input:  {x.shape}')
print(f'Output: {y.shape}')
print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

## Step 5: 开始训练

**tip**: 第一次建议先跑 `--epochs 5` 快速验证pipeline没问题, 再改回 100.

In [ ]:
# 快速验证 (5 epochs, 大约5-10分钟)
!python src/train_base.py --epochs 5

In [ ]:
# 完整训练 (100 epochs, 大约2-3小时)
# 跑完去掉下面注释再执行
# !python src/train_base.py --epochs 100

## Step 6: 验收标准

- `best_acc` 应该在 **93%-94%** 左右 (ResNet18-CIFAR 标准baseline)
- 低于 90% 可能是数据增强或LR schedule出了问题, 回来找我debug
- checkpoint保存在 `./checkpoints/resnet18_cifar_base.pth`

训完了就可以进 Stage 2: 实现 `ekf_pruner.py`.